<a href="https://colab.research.google.com/github/TheMadCatter150/hydra-sl-prediction/blob/main/HYDRA_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch==2.2.1+cu118 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.2.1+cu118.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.2.1+cu118.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.2.1+cu118.html
!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-2.2.1+cu118.html
!pip install torch-geometric
!pip install numpy==1.26.4 pandas scikit-learn scipy

Looking in indexes: https://download.pytorch.org/whl/cu118
Looking in links: https://data.pyg.org/whl/torch-2.2.1+cu118.html
Looking in links: https://data.pyg.org/whl/torch-2.2.1+cu118.html
Looking in links: https://data.pyg.org/whl/torch-2.2.1+cu118.html
Looking in links: https://data.pyg.org/whl/torch-2.2.1+cu118.html


In [15]:
from google.colab import drive
import os
import gc
import random
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch_geometric.nn import GATConv
from torch_geometric.data import Data

In [12]:
drive.mount('/content/drive')

# Override with an environment variable to point at your own copy of the
# data without editing this cell; defaults to the my original path.
DATA_DIR = os.environ.get('HYDRA_DATA_DIR', '/content/drive/MyDrive/Science Project/Colab Work/Files')
PPI_FILE_PATH = os.environ.get('HYDRA_PPI_FILE', '/content/drive/MyDrive/string_ppi_400.tsv')

sl_pairs = os.path.join(DATA_DIR, 'General SL', 'Human_SL.csv')
non_sl_pairs = os.path.join(DATA_DIR, 'General SL', 'Human_nonSL.csv')
sl_pairs_cleaned = os.path.join(DATA_DIR, 'General SL', 'sl_pairs_cleaned.csv')
non_sl_pairs_cleaned = os.path.join(DATA_DIR, 'General SL', 'non_sl_pairs_cleaned.csv')

sl_df = pd.read_csv(sl_pairs, sep=',')
non_sl_df = pd.read_csv(non_sl_pairs, sep=',')
sl_df.rename(columns={'n1.name': 'gene1', 'n2.name': 'gene2'}, inplace=True)
non_sl_df.rename(columns={'n1.name': 'gene1', 'n2.name': 'gene2'}, inplace=True)
sl_df.to_csv(sl_pairs_cleaned, index=False)
non_sl_df.to_csv(non_sl_pairs_cleaned, index=False)

BRCA_data_mutations = os.path.join(DATA_DIR, 'BRCA', 'data_mutations.txt')
kegg_canonical_pathways = os.path.join(DATA_DIR, 'KEGG', 'c2.cp.v2025.1.Hs.symbols.gmt')
ppi_file = PPI_FILE_PATH

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Main Methods

In [16]:
"""This is pretty much just methods equation 2"""
def build_features(tcga_mutation_file, kegg_pathway_file):

    #This is because I had a lot of errors
    with open(tcga_mutation_file) as f:
        skip = 0
        for line in f:
            if line.startswith('#'):
                skip += 1
            else:
                break

    tcga_df = pd.read_csv(tcga_mutation_file, sep='\t', skiprows=skip, low_memory=False)

    gene_col = next((c for c in tcga_df.columns
                      if c.strip().lower() in ('hugo_symbol', 'hugo symbol', 'gene', 'gene_symbol')), None)
    if gene_col is None:
        raise ValueError(f"Could not find a gene symbol col in {tcga_mutation_file}. "
                          f"Columns found: {list(tcga_df.columns)[:20]}")
    tcga_df = tcga_df.rename(columns={gene_col: 'Hugo_Symbol'})
    tcga_df['Hugo_Symbol'] = tcga_df['Hugo_Symbol'].astype(str).str.upper().str.strip()

    sample_col = next((c for c in tcga_df.columns
                        if c.strip().lower() in ('tumor_sample_barcode', 'sample_id', 'tumor_sample_id')), None)
    if sample_col is None:
        raise ValueError(f"Could not find a sample ID col in {tcga_mutation_file}. "
                          f"Columns found: {list(tcga_df.columns)[:20]}")
    tcga_df = tcga_df.rename(columns={sample_col: 'Tumor_Sample_Barcode'})

    n_patients = tcga_df['Tumor_Sample_Barcode'].nunique()
    silent_like = {'Silent', 'Intron', "3'UTR", "5'UTR", "5'Flank", "3'Flank",
                   'RNA', 'IGR', 'Splice_Region'}
    non_silent = tcga_df[~tcga_df['Variant_Classification'].isin(silent_like)]
    mut_freq = (non_silent.groupby('Hugo_Symbol')['Tumor_Sample_Barcode']
                .nunique().div(n_patients))
    all_tcga_genes = set(tcga_df['Hugo_Symbol'].unique())
    del tcga_df, non_silent
    gc.collect()

    gene_to_pathway_idx = defaultdict(set)
    pathway_list = []
    if kegg_pathway_file:
        with open(kegg_pathway_file, 'r') as f:
            for pidx, line in enumerate(f):
                parts = line.strip().split('\t')
                if len(parts) < 3:
                    continue
                pathway_list.append(parts[0])
                for gene in parts[2:]:
                    gene_to_pathway_idx[gene.upper().strip()].add(pidx)
    n_pathways = len(pathway_list)

    all_genes = sorted(all_tcga_genes | set(gene_to_pathway_idx.keys()))
    gene_idx = {g: i for i, g in enumerate(all_genes)}
    n_genes = len(all_genes)
    feat_dim = 1 + n_pathways

    X = np.zeros((n_genes, feat_dim), dtype=np.float32)
    mf = mut_freq.reindex(all_genes).fillna(0.0).to_numpy(dtype=np.float32)
    X[:, 0] = mf
    for gene, pidxs in gene_to_pathway_idx.items():
        gi = gene_idx.get(gene)
        if gi is None:
            continue
        idxs = np.fromiter(pidxs, dtype=np.int64)
        X[gi, 1 + idxs] = 1.0
    del gene_to_pathway_idx, mut_freq
    gc.collect()

    gene_feature_matrix = {g: X[i] for i, g in enumerate(all_genes)}
    print(f"Feat dim: {feat_dim} across {n_genes} genes "
          f"(1 mut freq + {n_pathways} KEGG pathway feat)")
    return gene_feature_matrix, all_genes, feat_dim


class GATEncoder(torch.nn.Module):
    """This one's just methods: GAT encoder"""

    def __init__(self, in_chan, hid_chan, out_chan, heads=4, dropout=0.3):
        super().__init__()
        self.gat1 = GATConv(in_chan, hid_chan, heads=heads, concat=True, dropout=dropout)
        self.gat2 = GATConv(hid_chan * heads, out_chan, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat2(x, edge_index)
        return F.normalize(x, p=2, dim=1)


class PairRegressor(torch.nn.Module):
    """This one I'm not even gonna explain its in the name"""
    def __init__(self, embed_dim, dropout=0.2):
        super().__init__()
        self.fc1 = torch.nn.Linear(embed_dim * 4, 128)
        self.fc2 = torch.nn.Linear(128, 64)
        self.fc3 = torch.nn.Linear(64, 1)
        self.dropout = torch.nn.Dropout(dropout)
        self.bn1 = torch.nn.BatchNorm1d(128)
        self.bn2 = torch.nn.BatchNorm1d(64)

    def forward(self, embeddings, pairs):
        h1 = embeddings[pairs[:, 0]]
        h2 = embeddings[pairs[:, 1]]
        combined = torch.cat([h1, h2, torch.abs(h1 - h2), h1 * h2], dim=1)
        out = self.dropout(F.gelu(self.bn1(self.fc1(combined))))
        out = self.dropout(F.gelu(self.bn2(self.fc2(out))))
        return self.fc3(out).view(-1)


def train_single_model(gat, reg, data_train, data_full,
                        train_pairs, train_labels, val_pairs, val_labels,
                        epochs=150, lr=0.005, patience_limit=20):
    params = list(gat.parameters()) + list(reg.parameters())
    optimizer = torch.optim.Adam(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10
    )
    best_val_auc = 0.5
    best_state = None
    patience = 0

    for epoch in range(epochs):
        gat.train()
        reg.train()
        optimizer.zero_grad()
        emb = gat(data_train.x, data_train.edge_index)
        preds = reg(emb, train_pairs)
        pos_weight = torch.tensor(
            [(train_labels == 0).sum().item() / max((train_labels == 1).sum().item(), 1)]
        ).to(train_pairs.device)
        loss = F.binary_cross_entropy_with_logits(preds, train_labels, pos_weight=pos_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        optimizer.step()

        if epoch % 5 == 0:
            gat.eval()
            reg.eval()
            with torch.no_grad():
                # Use the full graph for val embeddings, not data_train.
                # Val/test genes are inductively held out and have no edges in
                # data_train, so embedding them with data_train collapses AUC to ~0.5.
                val_emb = gat(data_full.x, data_full.edge_index)
                val_preds = reg(val_emb, val_pairs)
                val_probs = torch.sigmoid(val_preds)
                val_auc = roc_auc_score(val_labels.cpu().numpy(), val_probs.cpu().numpy())
            scheduler.step(val_auc)
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state = {
                    'gat': {k: v.clone() for k, v in gat.state_dict().items()},
                    'reg': {k: v.clone() for k, v in reg.state_dict().items()}
                }
                patience = 0
            else:
                patience += 1
                if patience >= patience_limit:
                    print(f"  Early stop at epoch {epoch}, best val AUC: {best_val_auc:.4f}")
                    break

    return best_state, best_val_auc

In [19]:
gene_feature_matrix, all_genes, feat_dim = build_features(BRCA_data_mutations, kegg_canonical_pathways)

sl_df = pd.read_csv(sl_pairs_cleaned)
non_sl_df = pd.read_csv(non_sl_pairs_cleaned)
sl_df['gene1'] = sl_df['gene1'].str.upper().str.strip()
sl_df['gene2'] = sl_df['gene2'].str.upper().str.strip()
non_sl_df['gene1'] = non_sl_df['gene1'].str.upper().str.strip()
non_sl_df['gene2'] = non_sl_df['gene2'].str.upper().str.strip()

if len(sl_df) > len(non_sl_df):
    sl_df = sl_df.sample(n=len(non_sl_df), random_state=42)
elif len(non_sl_df) > len(sl_df):
    non_sl_df = non_sl_df.sample(n=len(sl_df), random_state=42)

label_genes = (
    set(sl_df['gene1']) | set(sl_df['gene2']) |
    set(non_sl_df['gene1']) | set(non_sl_df['gene2'])
)
all_genes = [g for g in all_genes if g in label_genes]
gene_ID = {gene: idx for idx, gene in enumerate(all_genes)}
id_to_gene = {idx: gene for gene, idx in gene_ID.items()}
num_nodes = len(all_genes)

raw_x = torch.empty((num_nodes, feat_dim), dtype=torch.float32)
for gene, idx in gene_ID.items():
    raw_x[idx] = torch.from_numpy(gene_feature_matrix[gene])
del gene_feature_matrix
gc.collect()

ppi_edges_list = []
if ppi_file and os.path.exists(ppi_file):
    ppi_df = pd.read_csv(ppi_file, sep='\t', usecols=['gene1', 'gene2', 'combined_score'])
    ppi_df['gene1'] = ppi_df['gene1'].str.upper().str.strip()
    ppi_df['gene2'] = ppi_df['gene2'].str.upper().str.strip()
    filtered_ppi = ppi_df[
        ppi_df['gene1'].isin(gene_ID) &
        ppi_df['gene2'].isin(gene_ID) &
        (ppi_df['combined_score'] >= 400)
    ]
    g1_indices = filtered_ppi['gene1'].map(gene_ID).to_numpy()
    g2_indices = filtered_ppi['gene2'].map(gene_ID).to_numpy()
    ppi_edges_list = np.stack([g1_indices, g2_indices], axis=1).tolist()
    del ppi_df, filtered_ppi
    gc.collect()
print(f"PPI edges loaded: {len(ppi_edges_list)}")

# Gene-level inductive split (see Methods, Eq. 1)
random.seed(42)
unique_genes = list(all_genes)
random.shuffle(unique_genes)
n_genes = len(unique_genes)
n_val = int(n_genes * 0.15)
n_test = int(n_genes * 0.15)
val_genes = set(unique_genes[:n_val])
test_genes = set(unique_genes[n_val:n_val + n_test])
train_genes = set(unique_genes[n_val + n_test:])

sl_df['label'] = 1.0
non_sl_df['label'] = 0.0
combined_pairs_df = pd.concat([sl_df, non_sl_df], ignore_index=True)
combined_pairs_df['id1'] = combined_pairs_df['gene1'].map(gene_ID)
combined_pairs_df['id2'] = combined_pairs_df['gene2'].map(gene_ID)
combined_pairs_df = combined_pairs_df.dropna(subset=['id1', 'id2']).copy()
combined_pairs_df['id1'] = combined_pairs_df['id1'].astype(int)
combined_pairs_df['id2'] = combined_pairs_df['id2'].astype(int)

in_train1 = combined_pairs_df['gene1'].isin(train_genes)
in_train2 = combined_pairs_df['gene2'].isin(train_genes)
in_val1 = combined_pairs_df['gene1'].isin(val_genes)
in_val2 = combined_pairs_df['gene2'].isin(val_genes)
in_test1 = combined_pairs_df['gene1'].isin(test_genes)
in_test2 = combined_pairs_df['gene2'].isin(test_genes)

train_mask = in_train1 & in_train2
val_mask = (in_val1 | in_val2) & ~(in_test1 | in_test2)
test_mask = in_test1 | in_test2

train_df = combined_pairs_df[train_mask]
val_df = combined_pairs_df[val_mask]
test_df = combined_pairs_df[test_mask]

train_pairs = torch.tensor(train_df[['id1', 'id2']].to_numpy(), dtype=torch.long)
train_labels = torch.tensor(train_df['label'].to_numpy(), dtype=torch.float)
val_pairs = torch.tensor(val_df[['id1', 'id2']].to_numpy(), dtype=torch.long)
val_labels = torch.tensor(val_df['label'].to_numpy(), dtype=torch.float)
test_pairs = torch.tensor(test_df[['id1', 'id2']].to_numpy(), dtype=torch.long)
test_labels = torch.tensor(test_df['label'].to_numpy(), dtype=torch.float)
print(f"Pairs Train: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")

# Full (unsampled) SL pair list which DDGCN baseline need
sl_df_full = pd.read_csv(sl_pairs_cleaned)
sl_df_full['gene1'] = sl_df_full['gene1'].str.upper().str.strip()
sl_df_full['gene2'] = sl_df_full['gene2'].str.upper().str.strip()

del sl_df, non_sl_df, combined_pairs_df, train_df, val_df, test_df
gc.collect()


degrees = torch.zeros((num_nodes, 1), dtype=torch.float32)
if len(ppi_edges_list) > 0:
    edges_np = np.array(ppi_edges_list)
    np.add.at(degrees.numpy(), edges_np[:, 0], 1.0)
    np.add.at(degrees.numpy(), edges_np[:, 1], 1.0)

train_node_ids = [gene_ID[g] for g in train_genes]
train_mean = raw_x[train_node_ids].mean(dim=0)
train_std = raw_x[train_node_ids].std(dim=0) + 1e-6
scaled_x = (raw_x - train_mean) / train_std
X_final = torch.cat([scaled_x, degrees], dim=1)
del raw_x, scaled_x
gc.collect()


pos_train_mask = (train_labels == 1.0)
pos_train_pairs_list = train_pairs[pos_train_mask].cpu().numpy().tolist()
train_node_ids_set = set(gene_ID[g] for g in train_genes)

# Train HYDRA PPI-only 5-seed ensemble
full_edges_np = np.array(ppi_edges_list) if ppi_edges_list else np.zeros((0, 2), dtype=int)
if len(full_edges_np) > 0:
    full_edge_index = torch.tensor(ppi_edges_list, dtype=torch.long).t().contiguous()
    train_edge_mask = (
        np.isin(full_edges_np[:, 0], list(train_node_ids_set)) &
        np.isin(full_edges_np[:, 1], list(train_node_ids_set))
    )
    train_edges = full_edges_np[train_edge_mask].tolist()
    train_edge_index = (
        torch.tensor(train_edges, dtype=torch.long).t().contiguous()
        if train_edges else torch.zeros((2, 0), dtype=torch.long)
    )
else:
    full_edge_index = torch.zeros((2, 0), dtype=torch.long)
    train_edge_index = torch.zeros((2, 0), dtype=torch.long)


data_train_ppi = Data(x=X_final, edge_index=train_edge_index)
data_full_ppi = Data(x=X_final, edge_index=full_edge_index)
num_features = X_final.shape[1]

ppi_only_gat_list = []
ppi_only_reg_list = []
ppi_only_seed_aucs = []
saved_models = {}  # updated per-seed below so a partial run is still usable

for seed in range(5):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    gat_model = GATEncoder(in_chan=num_features, hid_chan=64, out_chan=32)
    regressor = PairRegressor(embed_dim=32)
    best_state, _ = train_single_model(
        gat_model, regressor, data_train_ppi, data_full_ppi,
        train_pairs, train_labels, val_pairs, val_labels,
        epochs=120, lr=0.005
    )
    if best_state:
        gat_model.load_state_dict(best_state['gat'])
        regressor.load_state_dict(best_state['reg'])
    gat_model.eval()
    regressor.eval()

    ppi_only_gat_list.append(gat_model)
    ppi_only_reg_list.append(regressor)

    with torch.no_grad():
        test_embeddings = gat_model(data_full_ppi.x, data_full_ppi.edge_index)
        test_predictions = regressor(test_embeddings, test_pairs)
        test_probs = torch.sigmoid(test_predictions).cpu().numpy()
    seed_auc = roc_auc_score(test_labels.cpu().numpy(), test_probs)
    ppi_only_seed_aucs.append(seed_auc)
    print(f"  seed {seed}: test AUC = {seed_auc:.4f}")

    # Update incrementally so a partial run still leaves a usable checkpoint
    saved_models['ppi_only'] = (ppi_only_gat_list, ppi_only_reg_list, data_full_ppi)

gc.collect()
print(f"\nppi_only ensemble AUC: {np.mean(ppi_only_seed_aucs):.4f} +/- {np.std(ppi_only_seed_aucs):.4f}")
print(f"Genes: {num_nodes}    Feature dim: {X_final.shape[1]}    PPI edges: {len(ppi_edges_list)}")

Feat dim: 4024 across 20245 genes (1 mut freq + 4023 KEGG pathway feat)
PPI edges loaded: 140458
Pairs Train: 2823, Val: 1424, Test: 1353


NameError: name 'Data' is not defined

In [10]:
"""
This is genuinely from DDGCN (Cai et al.) verbatim.
"""

import math
import random

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.nn.modules.module import Module
from torch.nn.parameter import Parameter
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score, average_precision_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == 'cuda' else " (no GPU found)"))


#Original model classes (from Cai et al. model.py)

class GraphConvolution(Module):
    def __init__(self, in_features, out_features, init, use_bias=True):
        super(GraphConvolution, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = Parameter(torch.FloatTensor(in_features, out_features))
        if use_bias:
            self.bias = Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.use_bias = use_bias
        self.reset_parameters(init)

    def reset_parameters(self, init):
        if init == 'Xavier':
            fan_in, fan_out = self.weight.shape
            init_range = np.sqrt(6.0 / (fan_in + fan_out))
            self.weight.data.uniform_(-init_range, init_range)
            if self.use_bias:
                torch.nn.init.constant_(self.bias, 0.)
        elif init == 'Kaiming':
            torch.nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
            if self.use_bias:
                fan_in, _ = self.weight.shape
                bound = 1 / math.sqrt(fan_in)
                torch.nn.init.uniform_(self.bias, -bound, bound)
        else:
            stdv = 1. / math.sqrt(self.weight.size(1))
            self.weight.data.uniform_(-stdv, stdv)
            if self.use_bias:
                self.bias.data.uniform_(-stdv, stdv)

    def forward(self, inputs, adj):
        if inputs.is_sparse:
            support = torch.sparse.mm(inputs, self.weight)
        else:
            support = torch.mm(inputs, self.weight)
        outputs = torch.sparse.mm(adj, support)
        if self.use_bias:
            return outputs + self.bias
        else:
            return outputs


class GCNEncoder(nn.Module):
    def __init__(self, nfeat, nhid1, nhid2, dropout, init, use_bias,
                 is_sparse_feat1, is_sparse_feat2):
        super(GCNEncoder, self).__init__()
        self.gc1 = GraphConvolution(nfeat, nhid1, init, use_bias)
        self.gc2 = GraphConvolution(nhid1, nhid2, init, use_bias)
        self.dropout = dropout
        self.is_sparse_feat1 = is_sparse_feat1
        self.is_sparse_feat2 = is_sparse_feat2

    def forward(self, x1, x2, adj):
        x1 = F.dropout(x1, self.dropout, training=self.training)
        x2 = F.dropout(x2, self.dropout, training=self.training)
        if self.is_sparse_feat1:
            x1 = x1.to_sparse()
        if self.is_sparse_feat2:
            x2 = x2.to_sparse()
        x1 = F.relu(self.gc1(x1, adj))
        x2 = F.relu(self.gc1(x2, adj))
        if self.training:
            mask = torch.bernoulli(
                x1.new(x1.size()).fill_(1 - self.dropout)) / (1 - self.dropout)
            x1 = x1 * mask
            x2 = x2 * mask
        x1 = self.gc2(x1, adj)
        x2 = self.gc2(x2, adj)
        return x1, x2


class InnerProductDecoder(nn.Module):
    def __init__(self, dropout):
        super(InnerProductDecoder, self).__init__()
        self.dropout = dropout

    def forward(self, inputs1, inputs2):
        if self.training:
            mask = torch.bernoulli(
                inputs1.new(inputs1.size()).fill_(1 - self.dropout)) / (1 - self.dropout)
            inputs1 = inputs1 * mask
            inputs2 = inputs2 * mask
        outputs1 = torch.mm(inputs1, inputs1.t())
        outputs2 = torch.mm(inputs2, inputs2.t())
        return outputs1, outputs2


class GraphAutoEncoder(nn.Module):
    def __init__(self, nfeat, nhid1, nhid2, dropout, init, use_bias,
                 is_sparse_feat1, is_sparse_feat2):
        super(GraphAutoEncoder, self).__init__()
        self.encoder = GCNEncoder(nfeat, nhid1, nhid2, dropout, init,
                                   use_bias, is_sparse_feat1, is_sparse_feat2)
        self.decoder = InnerProductDecoder(dropout)

    def forward(self, x1, x2, adj):
        e1, e2 = self.encoder(x1, x2, adj)
        return self.decoder(e1, e2)


#Original objective (from Cai et al. objective.py)

class ObjectiveFunction:
    def __init__(self, target_adj):
        num_edges = target_adj.sum()
        num_nodes = target_adj.shape[0]
        self.pos_weight = float(num_nodes ** 2 - num_edges) / num_edges
        self.norm = num_nodes ** 2 / float((num_nodes ** 2 - num_edges) * 2)
        self.target = target_adj

    def cal_loss(self, logit1, logit2, rho):
        loss1 = self.norm * binary_cross_entropy_with_logits(
            logit1, self.target,
            pos_weight=torch.tensor(self.pos_weight, device=logit1.device),
            reduction='mean')
        loss2 = self.norm * binary_cross_entropy_with_logits(
            logit2, self.target,
            pos_weight=torch.tensor(self.pos_weight, device=logit2.device),
            reduction='mean')
        return loss1 + rho * loss2


def build_sparse_adj_ddgcn(edge_list, n):
    if not edge_list:
        return torch.sparse.FloatTensor(
            torch.zeros((2, 0), dtype=torch.long),
            torch.zeros(0), torch.Size([n, n]))
    arr = np.array(edge_list)
    row, col = arr[:, 0], arr[:, 1]
    all_row = np.concatenate([row, col, np.arange(n)])
    all_col = np.concatenate([col, row, np.arange(n)])
    mat = sp.coo_matrix((np.ones(len(all_row), np.float32), (all_row, all_col)), shape=(n, n)).tocsr()
    deg = np.array(mat.sum(1)).flatten()
    inv_sqrt_deg = np.where(deg > 0, deg ** -0.5, 0.)
    normalized = sp.diags(inv_sqrt_deg).dot(mat).dot(sp.diags(inv_sqrt_deg)).tocoo().astype(np.float32)
    return torch.sparse.FloatTensor(
        torch.from_numpy(np.vstack((normalized.row, normalized.col)).astype(np.int64)),
        torch.from_numpy(normalized.data), torch.Size([n, n]))


def build_sl_adj_ddgcn(edge_list, n):
    mat = np.zeros((n, n), np.float32)
    for i, j in edge_list:
        mat[i, j] = mat[j, i] = 1.
    return torch.FloatTensor(mat)


def run_ddgcn_once(n_nodes, train_edges, test_pairs_np, test_labels_np,
                    label="", seed=0, epochs=2000, lr=0.01, nhid1=512, nhid2=256,
                    dropout=0.5, rho=1.0, tolerance_epoch=1000, stop_threshold=1e-5):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    feat1 = torch.eye(n_nodes).to(DEVICE)
    feat2 = build_sl_adj_ddgcn(train_edges, n_nodes).to(DEVICE)
    adj_norm = build_sparse_adj_ddgcn(train_edges, n_nodes).to(DEVICE)

    model = GraphAutoEncoder(n_nodes, nhid1, nhid2, dropout, 'Kaiming', False, True, True).to(DEVICE)
    adj_target = torch.zeros(n_nodes, n_nodes, dtype=torch.float32, device=DEVICE)
    for i, j in train_edges:
        adj_target[i, j] = 1.
        adj_target[j, i] = 1.
    objective = ObjectiveFunction(adj_target)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, amsgrad=True)

    last_loss = 1e9
    for epoch in range(epochs):
        model.train()
        logit1, logit2 = model(feat1, feat2, adj_norm)
        loss = objective.cal_loss(logit1, logit2, rho)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        converged = (epoch > tolerance_epoch and
                     abs((loss.item() - last_loss) / max(abs(last_loss), 1e-9)) < stop_threshold)
        if converged:
            break
        last_loss = loss.item()

    model.eval()
    with torch.no_grad():
        logit1, logit2 = model(feat1, feat2, adj_norm)
        r1 = torch.sigmoid(logit1).cpu().numpy()
        r2 = torch.sigmoid(logit2).cpu().numpy()
    score_matrix = np.power(r1 * np.power(r2, rho), 1. / (1. + rho))
    np.fill_diagonal(score_matrix, 0.)
    scores = score_matrix[test_pairs_np[:, 0], test_pairs_np[:, 1]]
    return (roc_auc_score(test_labels_np, scores),
            average_precision_score(test_labels_np, scores),
            score_matrix)


#Build DDGCN gene vocab (restricted to SL-annotated genes)
sl_id_pairs = []
for _, row in sl_df_full.iterrows():
    g1 = gene_ID.get(row['gene1'])
    g2 = gene_ID.get(row['gene2'])
    if g1 is not None and g2 is not None:
        sl_id_pairs.append([g1, g2])

sl_annotated_gene_ids = set()
for pair in sl_id_pairs:
    sl_annotated_gene_ids.update(pair)
sl_annotated_gene_list = sorted(sl_annotated_gene_ids)
remap = {old: new for new, old in enumerate(sl_annotated_gene_list)}
n_ddgcn = len(sl_annotated_gene_list)
remapped_sl_pairs = [[remap[p[0]], remap[p[1]]] for p in sl_id_pairs]
remapped_sl_set = set(map(tuple, remapped_sl_pairs))
print(f"DDGCN gene vocabulary: {n_ddgcn} SL-annotated genes "
      f"({len(gene_ID) - n_ddgcn} genes exclusive to HYDRA's full vocabulary)")

#Inductive gene-level evaluation (n=5 seeds)

pos_train_mask = (train_labels == 1.0)
train_pos_pairs = train_pairs[pos_train_mask].cpu().numpy()
test_pairs_np = test_pairs.cpu().numpy()
test_labels_np = test_labels.cpu().numpy()

inductive_train_edges = [
    [remap[int(p[0])], remap[int(p[1])]]
    for p in train_pos_pairs
    if int(p[0]) in remap and int(p[1]) in remap
]
inductive_test_pairs, inductive_test_labels = [], []
for pair, label in zip(test_pairs_np, test_labels_np):
    g1, g2 = int(pair[0]), int(pair[1])
    if g1 in remap and g2 in remap:
        inductive_test_pairs.append([remap[g1], remap[g2]])
        inductive_test_labels.append(label)
inductive_test_pairs = np.array(inductive_test_pairs)
inductive_test_labels = np.array(inductive_test_labels)

print(f"\nDDGCN inductive evaluation: {len(inductive_train_edges)} training edges, "
      f"{len(inductive_test_pairs)} test pairs")
ddgcn_inductive_aucs, ddgcn_inductive_auprcs = [], []
ddgcn_tst_score_seed = []
for seed in range(5):
    auc, auprc, score_matrix = run_ddgcn_once(
        n_ddgcn, inductive_train_edges, inductive_test_pairs, inductive_test_labels,
        label="inductive", seed=seed
    )
    ddgcn_inductive_aucs.append(auc)
    ddgcn_inductive_auprcs.append(auprc)
    ddgcn_tst_score_seed.append(score_matrix[inductive_test_pairs[:, 0], inductive_test_pairs[:, 1]])
    if seed == 0:
        ddgcn_score_matrix = score_matrix
    print(f"  seed {seed}: AUC={auc:.4f}  AUPRC={auprc:.4f}")

ddgcn_inductive_auc = np.mean(ddgcn_inductive_aucs)
ddgcn_inductive_auprc = np.mean(ddgcn_inductive_auprcs)
print(f"DDGCN inductive: AUC={ddgcn_inductive_auc:.4f} +/- {np.std(ddgcn_inductive_aucs):.4f}")

#Native pair-level evaluation (n=5 seeds) so we know its not bad because of me

random.seed(42)
shuffled_pairs = remapped_sl_pairs.copy()
random.shuffle(shuffled_pairs)
n_train = int(len(shuffled_pairs) * 0.8)
native_train_edges = shuffled_pairs[:n_train]
native_test_pos = np.array(shuffled_pairs[n_train:])

rng = np.random.RandomState(42)
native_test_neg = []
while len(native_test_neg) < len(native_test_pos):
    i, j = int(rng.randint(0, n_ddgcn)), int(rng.randint(0, n_ddgcn))
    if i != j and (i, j) not in remapped_sl_set and (j, i) not in remapped_sl_set:
        native_test_neg.append([i, j])
native_test_pairs = np.vstack([native_test_pos, np.array(native_test_neg)])
native_test_labels = np.array([1.] * len(native_test_pos) + [0.] * len(native_test_neg))

print(f"\nDDGCN native pair-level evaluation: {len(native_train_edges)} training edges, "
      f"{len(native_test_pairs)} test pairs")
ddgcn_native_aucs, ddgcn_native_auprcs = [], []
for seed in range(5):
    auc, auprc, _ = run_ddgcn_once(
        n_ddgcn, native_train_edges, native_test_pairs, native_test_labels,
        label="native", seed=seed
    )
    ddgcn_native_aucs.append(auc)
    ddgcn_native_auprcs.append(auprc)
    print(f"  seed {seed}: AUC={auc:.4f}  AUPRC={auprc:.4f}")

ddgcn_native_auc = np.mean(ddgcn_native_aucs)
print(f"DDGCN native: AUC={ddgcn_native_auc:.4f} +/- {np.std(ddgcn_native_aucs):.4f}")

# summary
t_native_vs_inductive, p_native_vs_inductive = scipy_stats.ttest_rel(ddgcn_native_aucs, ddgcn_inductive_aucs)
print(f"\nNative vs. inductive AUC: t={t_native_vs_inductive:.3f}, p={p_native_vs_inductive:.2e}")
print(f"Ran on: {DEVICE}")

Device: cuda (Tesla T4)
DDGCN gene vocabulary: 2695 SL-annotated genes (16 genes exclusive to HYDRA's full vocabulary)

DDGCN inductive evaluation: 1420 training edges, 1326 test pairs


/tmp/ipykernel_2080/741206232.py:140: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight=torch.tensor(self.pos_weight, device=logit1.device),
/tmp/ipykernel_2080/741206232.py:144: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight=torch.tensor(self.pos_weight, device=logit2.device),


  seed 0: AUC=0.5000  AUPRC=0.5098


/tmp/ipykernel_2080/741206232.py:140: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight=torch.tensor(self.pos_weight, device=logit1.device),
/tmp/ipykernel_2080/741206232.py:144: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight=torch.tensor(self.pos_weight, device=logit2.device),


  seed 1: AUC=0.4970  AUPRC=0.5096


...

  seed 2: AUC=0.9239  AUPRC=0.9461


/tmp/ipykernel_2080/741206232.py:140: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight=torch.tensor(self.pos_weight, device=logit1.device),
/tmp/ipykernel_2080/741206232.py:144: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight=torch.tensor(self.pos_weight, device=logit2.device),


  seed 3: AUC=0.9206  AUPRC=0.9447


/tmp/ipykernel_2080/741206232.py:140: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight=torch.tensor(self.pos_weight, device=logit1.device),
/tmp/ipykernel_2080/741206232.py:144: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight=torch.tensor(self.pos_weight, device=logit2.device),


  seed 4: AUC=0.9198  AUPRC=0.9443
DDGCN native: AUC=0.9214 +/- 0.0018

Native vs. inductive AUC: t=511.251, p=8.78e-11
Ran on: cuda


In [7]:
"""The ablation cell"""
from collections import Counter

import numpy as np
import pandas as pd
import torch
import random
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.data import Data


def build_graph_pair(edges, train_node_ids_set, X_final):
    """Build (train-only, full-vocabulary) graph pairs for one edge condition."""
    if len(edges) > 0:
        full_edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edges_np = np.array(edges)
        train_edge_mask = (
            np.isin(edges_np[:, 0], list(train_node_ids_set)) &
            np.isin(edges_np[:, 1], list(train_node_ids_set))
        )
        train_edges = edges_np[train_edge_mask].tolist()
        train_edge_index = (
            torch.tensor(train_edges, dtype=torch.long).t().contiguous()
            if train_edges else torch.zeros((2, 0), dtype=torch.long)
        )
    else:
        full_edge_index = torch.zeros((2, 0), dtype=torch.long)
        train_edge_index = torch.zeros((2, 0), dtype=torch.long)
    return (Data(x=X_final, edge_index=train_edge_index),
            Data(x=X_final, edge_index=full_edge_index))


def evaluate_existing_ensemble(gat_list, reg_list, data_full, test_pairs, test_labels):
    """Inference-only pass over already-trained models (no retraining)."""
    test_labels_np = test_labels.cpu().numpy()
    seed_aucs, seed_auprcs, all_test_probs = [], [], []
    for gat, reg in zip(gat_list, reg_list):
        gat.eval()
        reg.eval()
        with torch.no_grad():
            test_embeddings = gat(data_full.x, data_full.edge_index)
            test_predictions = reg(test_embeddings, test_pairs)
            test_probs = torch.sigmoid(test_predictions).cpu().numpy()
        seed_aucs.append(roc_auc_score(test_labels_np, test_probs))
        seed_auprcs.append(average_precision_score(test_labels_np, test_probs))
        all_test_probs.append(test_probs)
    return seed_aucs, seed_auprcs, all_test_probs


graph_conditions = ["ppi_only", "sl_edges", "ppi_plus_sl"]
results_rows = []

if 'saved_models' not in dir():
    saved_models = {}

for graph_mode in graph_conditions:
    print(f"\n{graph_mode}")

    # Reuse the main training cell's ppi_only ensemble if it's already trained.
    if (graph_mode == "ppi_only" and 'ppi_only' in saved_models
            and len(saved_models['ppi_only'][0]) >= 5):
        print("  Reusing existing ppi_only ensemble (5 seeds) and skipping retrain.")
        gat_list, reg_list, data_full = saved_models['ppi_only']
        seed_aucs, seed_auprcs, all_test_probs = evaluate_existing_ensemble(
            gat_list, reg_list, data_full, test_pairs, test_labels
        )
        for seed, (auc, auprc) in enumerate(zip(seed_aucs, seed_auprcs)):
            print(f"  seed {seed} (reused): AUC={auc:.4f}  AUPRC={auprc:.4f}")

    else:
        if graph_mode == "ppi_only":
            edges = list(ppi_edges_list)
        elif graph_mode == "sl_edges":
            edges = list(pos_train_pairs_list)
        else:  # ppi_plus_sl
            edges = list(ppi_edges_list) + list(pos_train_pairs_list)

        data_train, data_full = build_graph_pair(edges, train_node_ids_set, X_final)
        num_features = X_final.shape[1]
        seed_aucs, seed_auprcs, all_test_probs = [], [], []
        gat_list, reg_list = [], []

        for seed in range(5):
            torch.manual_seed(seed)
            np.random.seed(seed)
            random.seed(seed)
            gat_model = GATEncoder(in_chan=num_features, hid_chan=64, out_chan=32)
            regressor = PairRegressor(embed_dim=32)
            best_state, _ = train_single_model(
                gat_model, regressor, data_train, data_full,
                train_pairs, train_labels, val_pairs, val_labels,
                epochs=120, lr=0.005
            )
            if best_state is None:
                print(f"  seed {seed}: no improvement, skipping.")
                continue
            gat_model.load_state_dict(best_state['gat'])
            regressor.load_state_dict(best_state['reg'])
            gat_model.eval()
            regressor.eval()
            with torch.no_grad():
                test_embeddings = gat_model(data_full.x, data_full.edge_index)
                test_predictions = regressor(test_embeddings, test_pairs)
                test_probs = torch.sigmoid(test_predictions).cpu().numpy()
            test_labels_np = test_labels.cpu().numpy()
            auc = roc_auc_score(test_labels_np, test_probs)
            auprc = average_precision_score(test_labels_np, test_probs)
            seed_aucs.append(auc)
            seed_auprcs.append(auprc)
            all_test_probs.append(test_probs)
            gat_list.append(gat_model)
            reg_list.append(regressor)
            print(f"  seed {seed}: AUC={auc:.4f}  AUPRC={auprc:.4f}")

    probability_ensemble_probs = np.mean(all_test_probs, axis=0)
    test_labels_np = test_labels.cpu().numpy()
    probability_ensemble_auc = roc_auc_score(test_labels_np, probability_ensemble_probs)
    probability_ensemble_auprc = average_precision_score(test_labels_np, probability_ensemble_probs)

    mean_seed_auc = float(np.mean(seed_aucs))
    std_seed_auc = float(np.std(seed_aucs))
    mean_seed_auprc = float(np.mean(seed_auprcs))
    std_seed_auprc = float(np.std(seed_auprcs))

    saved_models[graph_mode] = (gat_list, reg_list, data_full)
    results_rows.append({
        'graph_mode': graph_mode,
        'mean_seed_auc': round(mean_seed_auc, 4),
        'std_seed_auc': round(std_seed_auc, 4),
        'mean_seed_auprc': round(mean_seed_auprc, 4),
        'std_seed_auprc': round(std_seed_auprc, 4),
        'seed_aucs': seed_aucs,
        'seed_auprcs': seed_auprcs,
        'probability_ensemble_auc': round(probability_ensemble_auc, 4),
        'probability_ensemble_auprc': round(probability_ensemble_auprc, 4),
        'probability_ensemble_probs': probability_ensemble_probs,
    })
    print(f"  [{graph_mode}] mean +/- SD: AUC={mean_seed_auc:.4f}+/-{std_seed_auc:.4f}  "
          f"AUPRC={mean_seed_auprc:.4f}+/-{std_seed_auprc:.4f}")

results_df = pd.DataFrame(results_rows)
print("\n" + results_df[['graph_mode', 'mean_seed_auc', 'std_seed_auc',
                          'mean_seed_auprc', 'std_seed_auprc']].to_string(index=False))
results_df.to_csv(os.path.join(DATA_DIR, '..', 'HYDRA_three_condition_results.csv'), index=False)

ppi_aucs = results_df.loc[results_df['graph_mode'] == 'ppi_only', 'seed_aucs'].values[0]
sl_aucs = results_df.loc[results_df['graph_mode'] == 'sl_edges', 'seed_aucs'].values[0]
psl_aucs = results_df.loc[results_df['graph_mode'] == 'ppi_plus_sl', 'seed_aucs'].values[0]

t1, p1 = scipy_stats.ttest_rel(ppi_aucs, psl_aucs)
t2, p2 = scipy_stats.ttest_rel(ppi_aucs, sl_aucs)
t3, p3 = scipy_stats.ttest_rel(sl_aucs, psl_aucs)
print(f"\nppi_only vs ppi_plus_sl : t={t1:.3f}, p={p1:.4f}")
print(f"ppi_only vs sl_edges    : t={t2:.3f}, p={p2:.4f}")
print(f"sl_edges vs ppi_plus_sl : t={t3:.3f}, p={p3:.4f}")

# Degree-stratified AUC
sl_degree = Counter()
for pair, label in zip(train_pairs.tolist(), train_labels.tolist()):
    if int(label) == 1:
        sl_degree[pair[0]] += 1
        sl_degree[pair[1]] += 1


def sl_bucket(degree):
    if degree == 0:
        return "0_unseen"
    elif degree <= 2:
        return "1-2"
    elif degree <= 5:
        return "3-5"
    else:
        return "6+"


test_pairs_np = test_pairs.cpu().numpy()
test_labels_np = test_labels.cpu().numpy()
pair_buckets = np.array([
    sl_bucket(min(sl_degree.get(int(g1), 0), sl_degree.get(int(g2), 0)))
    for g1, g2 in test_pairs_np
])

strat_rows = []
for row in results_rows:
    mode = row['graph_mode']
    probs = row['probability_ensemble_probs']
    for bucket in ["0_unseen", "1-2", "3-5", "6+"]:
        mask = pair_buckets == bucket
        if mask.sum() < 5:
            continue
        bucket_labels = test_labels_np[mask]
        if len(np.unique(bucket_labels)) < 2:
            continue
        strat_rows.append({
            'graph_mode': mode, 'sl_degree_bucket': bucket,
            'n_pairs': int(mask.sum()),
            'pos_frac': round(float(bucket_labels.mean()), 3),
            'auc': round(roc_auc_score(bucket_labels, probs[mask]), 4),
            'auprc': round(average_precision_score(bucket_labels, probs[mask]), 4),
        })
strat_df = pd.DataFrame(strat_rows)
print("\n" + strat_df.to_string(index=False))
strat_df.to_csv(os.path.join(DATA_DIR, '..', 'HYDRA_degree_stratification.csv'), index=False)


ppi_only
  Reusing existing ppi_only ensemble (5 seeds) and skipping retrain.
  seed 0 (reused): AUC=0.8400  AUPRC=0.8598
  seed 1 (reused): AUC=0.6768  AUPRC=0.6885
  seed 2 (reused): AUC=0.8515  AUPRC=0.8404
  seed 3 (reused): AUC=0.7119  AUPRC=0.7259
  seed 4 (reused): AUC=0.8035  AUPRC=0.7960
  [ppi_only] mean +/- SD: AUC=0.7767+/-0.0700  AUPRC=0.7821+/-0.0656

sl_edges
  seed 0: AUC=0.8533  AUPRC=0.8633
  seed 1: AUC=0.8315  AUPRC=0.8317
  seed 2: AUC=0.8533  AUPRC=0.8562
  seed 3: AUC=0.8384  AUPRC=0.8473
  seed 4: AUC=0.8482  AUPRC=0.8547
  [sl_edges] mean +/- SD: AUC=0.8449+/-0.0086  AUPRC=0.8506+/-0.0108

ppi_plus_sl
  seed 0: AUC=0.6765  AUPRC=0.6581
  seed 1: AUC=0.8129  AUPRC=0.8083
  seed 2: AUC=0.6704  AUPRC=0.6948
  seed 3: AUC=0.8117  AUPRC=0.8243
  seed 4: AUC=0.8334  AUPRC=0.8474
  [ppi_plus_sl] mean +/- SD: AUC=0.7610+/-0.0719  AUPRC=0.7666+/-0.0755

 graph_mode  mean_seed_auc  std_seed_auc  mean_seed_auprc  std_seed_auprc
   ppi_only         0.7767        0.0700   

In [ ]:

import os

import numpy as np
import pandas as pd
import torch

OUTPUT_DIR = os.path.join(DATA_DIR, '..')
COSMIC_CGC_FILE = os.path.join(DATA_DIR, 'cosmic_cgc.tsv')
DDGCN_NOISE_THRESHOLD = 0.55
SCORE_CHUNK_SIZE = 50000


def load_cosmic_cgc(filepath):
    if os.path.exists(filepath):
        try:
            df = pd.read_csv(filepath, sep='\t')
            col = next((c for c in df.columns if 'Gene Symbol' in c or 'gene_symbol' in c.lower()), None)
            if col:
                genes = set(df[col].dropna().str.upper().str.strip())
                print(f"Loaded {len(genes)} COSMIC CGC genes from file.")
                return genes
        except Exception as e:
            print(f"Could not read COSMIC file: {e}")
    print("COSMIC CGC file not found using curated fallback gene set.")
    return None


_cosmic_from_file = load_cosmic_cgc(COSMIC_CGC_FILE)
COSMIC_CGC_FALLBACK = {
    'ABL1', 'ABL2', 'ACVR1', 'AFF1', 'AKT1', 'AKT2', 'AKT3', 'ALK', 'AMER1', 'APC',
    'ARID1A', 'ARID1B', 'ARID2', 'ASXL1', 'ASXL2', 'ATM', 'ATR', 'ATRX', 'AXIN1',
    'AXIN2', 'BAP1', 'BCL2', 'BCL6', 'BRAF', 'BRCA1', 'BRCA2', 'BRD4', 'BRIP1',
    'BTK', 'CALR', 'CARD11', 'CASP8', 'CBFB', 'CBL', 'CCND1', 'CCND2', 'CCND3',
    'CCNE1', 'CDH1', 'CDK12', 'CDK4', 'CDK6', 'CDKN1A', 'CDKN1B', 'CDKN2A',
    'CDKN2B', 'CEBPA', 'CHEK2', 'CIC', 'CREBBP', 'CRTC1', 'CSF1R', 'CTCF',
    'CTNNB1', 'DAXX', 'DNMT3A', 'EGFR', 'EZH2', 'FANCA', 'FANCC', 'FANCD2',
    'FANCE', 'FANCF', 'FANCG', 'FANCL', 'FBXW7', 'FH', 'FLI1', 'FLT3', 'FOXA1',
    'FOXL2', 'FOXO1', 'FOXP1', 'FUS', 'GATA1', 'GATA2', 'GATA3', 'GNA11', 'GNAQ',
    'GNAS', 'HMGA1', 'HMGA2', 'HRAS', 'IDH1', 'IDH2', 'IKZF1', 'JAK1', 'JAK2',
    'JAK3', 'KDM5C', 'KDM6A', 'KIT', 'KMT2A', 'KMT2C', 'KMT2D', 'KRAS', 'LATS1',
    'LATS2', 'MAP2K1', 'MAP2K2', 'MAP2K4', 'MAP3K1', 'MAPK1', 'MAX', 'MCL1', 'MDM2',
    'MDM4', 'MED12', 'MEN1', 'MET', 'MLH1', 'MPL', 'MSH2', 'MSH6', 'MYC', 'MYCN',
    'MYD88', 'NF1', 'NF2', 'NFE2L2', 'NOTCH1', 'NOTCH2', 'NPM1', 'NRAS', 'NSD1',
    'NSD2', 'NTRK1', 'NTRK2', 'NTRK3', 'PALB2', 'PAX5', 'PDGFRA', 'PDGFRB',
    'PIK3CA', 'PIK3R1', 'PML', 'PMS2', 'POLE', 'PTEN', 'PTPN11', 'RAF1', 'RB1',
    'RET', 'RHOA', 'RNF43', 'ROS1', 'RUNX1', 'SETD2', 'SF3B1', 'SMAD2', 'SMAD3',
    'SMAD4', 'SMARCA4', 'SMARCB1', 'SMO', 'SPOP', 'SRC', 'STAG2', 'STAT3', 'STAT5B',
    'STK11', 'SUZ12', 'SYK', 'TET1', 'TET2', 'TP53', 'TSC1', 'TSC2', 'VHL', 'WRN',
    'WT1', 'XPA', 'XPC', 'ZEB1', 'ZRSR2',
    'ACVR2A', 'ATF7IP', 'B2M', 'BAX', 'BBC3', 'BRD7', 'BTG2', 'CD22', 'CDK8', 'CDK9',
    'CFL2', 'CKS1B', 'DAPK1', 'DAPK2', 'DDX3X', 'DOCK8', 'ELK4', 'EXT1', 'FHIT',
    'HGF', 'HOXA3', 'IGF1R', 'INHBA', 'INSR', 'IRF1', 'IRF2', 'LMNA',
    'LRP1B', 'MAP3K14', 'MCM3AP', 'MEF2B', 'MNX1', 'MUC16', 'MUTYH', 'NCOA1',
    'NCOA2', 'PTPRD', 'QKI', 'RAD17', 'RAD51', 'RAD51C', 'RAD51D', 'RAD54L',
    'RB1CC1', 'RBM10', 'RECQL', 'RIF1', 'RNF168', 'RNF8', 'ROBO2', 'RRM2B',
    'SETD7', 'SETDB1', 'SF3A1', 'SIRT1', 'TERT', 'TGFBR1', 'TGFBR2', 'TLE4',
    'TP53BP1', 'TP73', 'TRAF3', 'TRIM37', 'TRRAP', 'TWIST1', 'TYRO3', 'UBR5',
    'UBTF', 'VAV1', 'VEGFA', 'WEE1', 'XRCC2', 'XRCC3', 'ZEB2', 'ZNF217',
}
COSMIC_CGC = _cosmic_from_file if _cosmic_from_file else COSMIC_CGC_FALLBACK

DDR = {
    'BRCA1', 'BRCA2', 'RAD51', 'RAD51B', 'RAD51C', 'RAD51D', 'RAD52', 'RAD54L',
    'XRCC2', 'XRCC3', 'PALB2', 'FANCD2', 'FANCA', 'FANCC', 'FANCE', 'FANCF',
    'FANCG', 'FANCL', 'BRIP1', 'NBN', 'MRE11', 'RAD50', 'BLM', 'RECQL', 'RECQL4',
    'WRN', 'PRKDC', 'XRCC4', 'XRCC5', 'XRCC6', 'LIG4', 'PARP1', 'PARP2', 'PARP3',
    'XRCC1', 'LIG3', 'POLB', 'FEN1', 'OGG1', 'MUTYH', 'MLH1', 'MSH2', 'MSH6',
    'PMS2', 'MSH3', 'EXO1', 'RFC1', 'PCNA', 'XPC', 'XPA', 'DDB1', 'DDB2', 'ATM',
    'ATR', 'CHEK1', 'CHEK2', 'TP53', 'TP53BP1', 'MDM2', 'MDM4', 'WEE1', 'RAD17',
    'POLE', 'POLD1', 'POLD2', 'BARD1', 'RNF8', 'RNF168', 'RIF1', 'SLX4', 'HELB',
}
ALL_CANCER = COSMIC_CGC | DDR | {
    'BCL2L1', 'MCL1', 'BAK1', 'BID', 'CASP3', 'CASP7', 'CASP8', 'CASP9', 'APAF1',
    'CCND1', 'CCND2', 'CCND3', 'CCNE1', 'CDK1', 'CDK2', 'CDK4', 'CDK6', 'CDKN1A',
    'CDKN1B', 'CDKN2A', 'RB1', 'PLK1', 'AURKA', 'AURKB', 'BUB1', 'WEE1', 'TTK',
    'PIK3CA', 'PIK3CB', 'PIK3R1', 'AKT1', 'AKT2', 'AKT3', 'PTEN', 'MTOR', 'TSC1',
    'TSC2', 'KRAS', 'NRAS', 'HRAS', 'BRAF', 'RAF1', 'MAP2K1', 'MAP2K2', 'MAPK1',
    'MAPK3', 'NF1', 'SPRY2', 'DUSP6',
}


def annotate_gene(gene):
    gene = gene.upper()
    tags = []
    if gene in COSMIC_CGC:
        tags.append("COSMIC CGC")
    if gene in DDR:
        tags.append("DNA repair")
    return "; ".join(tags) if tags else "-"


#Score all novel pairs with the trained HYDRA ensemble

if 'saved_models' in dir() and 'ppi_only' in saved_models:
    gat_list, reg_list, data_full_ppi = saved_models['ppi_only']
    print(f"Using ppi_only ensemble ({len(gat_list)} seed(s)).")
else:
    raise RuntimeError("saved_models['ppi_only'] not found but we ball.")

known_pairs = set()
for _, row in sl_df_full.iterrows():
    g1 = gene_ID.get(row['gene1'])
    g2 = gene_ID.get(row['gene2'])
    if g1 is not None and g2 is not None:
        known_pairs.add((min(g1, g2), max(g1, g2)))

n_genes = len(all_genes)
print(f"Genes: {n_genes}  Known SL pairs excluded: {len(known_pairs)}")

all_embeddings = []
for gat, reg in zip(gat_list, reg_list):
    gat.eval()
    reg.eval()
    with torch.no_grad():
        all_embeddings.append(gat(data_full_ppi.x, data_full_ppi.edge_index))

results = []
candidate_chunk = []
for i in range(n_genes):
    for j in range(i + 1, n_genes):
        if (i, j) not in known_pairs:
            candidate_chunk.append((i, j))
        if len(candidate_chunk) >= SCORE_CHUNK_SIZE:
            chunk_tensor = torch.tensor(candidate_chunk, dtype=torch.long)
            with torch.no_grad():
                chunk_logits = [reg(emb, chunk_tensor).cpu().numpy() for emb, reg in zip(all_embeddings, reg_list)]
            avg_logits = np.mean(chunk_logits, axis=0)
            for idx, (g1, g2) in enumerate(candidate_chunk):
                results.append({
                    'gene1': id_to_gene[g1], 'gene2': id_to_gene[g2],
                    'hydra_logit': float(avg_logits[idx]),
                    'hydra_score': float(1 / (1 + np.exp(-np.clip(avg_logits[idx], -500, 500)))),
                    'gene1_id': g1, 'gene2_id': g2,
                })
            candidate_chunk = []

if candidate_chunk:
    chunk_tensor = torch.tensor(candidate_chunk, dtype=torch.long)
    with torch.no_grad():
        chunk_logits = [reg(emb, chunk_tensor).cpu().numpy() for emb, reg in zip(all_embeddings, reg_list)]
    avg_logits = np.mean(chunk_logits, axis=0)
    for idx, (g1, g2) in enumerate(candidate_chunk):
        results.append({
            'gene1': id_to_gene[g1], 'gene2': id_to_gene[g2],
            'hydra_logit': float(avg_logits[idx]),
            'hydra_score': float(1 / (1 + np.exp(-np.clip(avg_logits[idx], -500, 500)))),
            'gene1_id': g1, 'gene2_id': g2,
        })

pred_df = pd.DataFrame(results).sort_values('hydra_logit', ascending=False).reset_index(drop=True)
print(f"Scored {len(pred_df):,} novel pairs. Logit range: "
      f"{pred_df['hydra_logit'].min():.3f} to {pred_df['hydra_logit'].max():.3f}")


def get_ddgcn_score(g1_id, g2_id):
    if g1_id not in remap or g2_id not in remap:
        return None
    return float(ddgcn_score_matrix[remap[g1_id], remap[g2_id]])


pred_df['ddgcn_score'] = pred_df.apply(lambda r: get_ddgcn_score(r['gene1_id'], r['gene2_id']), axis=1)
pred_df['ddgcn_can_score'] = pred_df['ddgcn_score'].notna()
pred_df['ddgcn_random'] = pred_df['ddgcn_can_score'] & (pred_df['ddgcn_score'] < DDGCN_NOISE_THRESHOLD)
pred_df['ddgcn_outside_vocab'] = ~pred_df['ddgcn_can_score']
pred_df['exclusively_hydra'] = pred_df['ddgcn_outside_vocab'] | pred_df['ddgcn_random']
pred_df['hydra_wins_ddgcn'] = (
    pred_df['ddgcn_can_score']
    & (pred_df['ddgcn_score'] >= DDGCN_NOISE_THRESHOLD)
    & (pred_df['hydra_score'] > pred_df['ddgcn_score'])
)
print(f"Exclusively HYDRA: {pred_df['exclusively_hydra'].sum():,} pairs  |  "
      f"HYDRA > DDGCN (real signal): {pred_df['hydra_wins_ddgcn'].sum():,} pairs")

# Checkpoint that never worked when I was debugging but might now
pred_df.to_parquet(os.path.join(OUTPUT_DIR, 'HYDRA_full_scored_pairs.parquet'), index=False)

#Annotate against the curated cancer gene set

pred_df['ann1'] = pred_df['gene1'].apply(annotate_gene)
pred_df['ann2'] = pred_df['gene2'].apply(annotate_gene)
pred_df['any_cancer'] = pred_df['gene1'].str.upper().isin(ALL_CANCER) | pred_df['gene2'].str.upper().isin(ALL_CANCER)
pred_df['both_cosmic'] = pred_df['gene1'].str.upper().isin(COSMIC_CGC) & pred_df['gene2'].str.upper().isin(COSMIC_CGC)
pred_df['any_ddr'] = pred_df['gene1'].str.upper().isin(DDR) | pred_df['gene2'].str.upper().isin(DDR)

cancer_df = pred_df[pred_df['any_cancer']].sort_values('hydra_logit', ascending=False)
ddr_df = pred_df[pred_df['any_ddr']].sort_values('hydra_logit', ascending=False)
both_cosmic_df = pred_df[pred_df['both_cosmic']].sort_values('hydra_logit', ascending=False)
lit_df = pred_df[pred_df['any_ddr'] | pred_df['both_cosmic']].sort_values('hydra_logit', ascending=False).head(50)

#Summary statistics used in the manuscript

ddgcn_scoreable = pred_df.loc[pred_df['ddgcn_can_score'], 'ddgcn_score']
hydra_scores = pred_df['hydra_score']
summary_stats = {
    'total_novel_pairs': len(pred_df),
    'exclusively_hydra': int(pred_df['exclusively_hydra'].sum()),
    'any_cancer': int(pred_df['any_cancer'].sum()),
    'exclusively_hydra_cancer': int((pred_df['exclusively_hydra'] & pred_df['any_cancer']).sum()),
    'both_cosmic': int(pred_df['both_cosmic'].sum()),
    'any_ddr': int(pred_df['any_ddr'].sum()),
    'exclusively_hydra_ddr': int((pred_df['exclusively_hydra'] & pred_df['any_ddr']).sum()),
    'hydra_mean_score': round(float(hydra_scores.mean()), 4),
    'hydra_sd_score': round(float(hydra_scores.std()), 4),
    'hydra_pct_above_070': round(float((hydra_scores > 0.70).mean() * 100), 1),
    'ddgcn_mean_score': round(float(ddgcn_scoreable.mean()), 4),
    'ddgcn_sd_score': round(float(ddgcn_scoreable.std()), 4),
    'ddgcn_pct_below_055': round(float((ddgcn_scoreable < 0.55).mean() * 100), 1),
    'logit_min': round(float(pred_df['hydra_logit'].min()), 3),
    'logit_max': round(float(pred_df['hydra_logit'].max()), 3),
}
for key, value in summary_stats.items():
    print(f"  {key:<28} {value}")
pd.DataFrame([summary_stats]).to_csv(os.path.join(OUTPUT_DIR, 'HYDRA_paper_numbers.csv'), index=False)

pred_df.head(5000).to_csv(os.path.join(OUTPUT_DIR, 'HYDRA_top5000_predictions.csv'), index=False)
cancer_df.head(1000).to_csv(os.path.join(OUTPUT_DIR, 'HYDRA_cancer_vs_DDGCN.csv'), index=False)
ddr_df.head(100).to_csv(os.path.join(OUTPUT_DIR, 'HYDRA_DDR_predictions.csv'), index=False)
pred_df[pred_df['exclusively_hydra'] & pred_df['any_cancer']].head(500).to_csv(
    os.path.join(OUTPUT_DIR, 'HYDRA_exclusively_HYDRA_cancer.csv'), index=False)
lit_df.to_csv(os.path.join(OUTPUT_DIR, 'HYDRA_literature_supported.csv'), index=False)

In [ ]:
"""Cross cohort validation cell"""

import os
import gc
import random

import numpy as np
import pandas as pd
import torch
from scipy import stats
from sklearn.metrics import roc_auc_score
from torch_geometric.data import Data

METABRIC_PATH = os.path.join(DATA_DIR, 'brca_metabric_extracted', 'brca_metabric', 'data_mutations.txt')
assert BRCA_data_mutations != METABRIC_PATH, "TCGA and METABRIC paths must differ."


def run_hydra_on_cohort(mutation_file, cohort_name, n_seeds=5):
    print(f"\n{cohort_name}")
    gene_feature_matrix, cohort_genes, feat_dim = build_features(mutation_file, kegg_canonical_pathways)

    sl = pd.read_csv(sl_pairs_cleaned)
    non_sl = pd.read_csv(non_sl_pairs_cleaned)
    sl['gene1'] = sl['gene1'].str.upper().str.strip()
    sl['gene2'] = sl['gene2'].str.upper().str.strip()
    non_sl['gene1'] = non_sl['gene1'].str.upper().str.strip()
    non_sl['gene2'] = non_sl['gene2'].str.upper().str.strip()
    if len(sl) > len(non_sl):
        sl = sl.sample(n=len(non_sl), random_state=42)
    elif len(non_sl) > len(sl):
        non_sl = non_sl.sample(n=len(sl), random_state=42)

    label_genes = set(sl['gene1']) | set(sl['gene2']) | set(non_sl['gene1']) | set(non_sl['gene2'])
    cohort_genes = [g for g in cohort_genes if g in label_genes]
    cohort_gene_ID = {g: i for i, g in enumerate(cohort_genes)}
    n_genes = len(cohort_genes)

    raw_x = torch.empty((n_genes, feat_dim), dtype=torch.float32)
    for gene, idx in cohort_gene_ID.items():
        raw_x[idx] = torch.from_numpy(gene_feature_matrix[gene])
    del gene_feature_matrix
    gc.collect()

    ppi_edges = []
    if ppi_file and os.path.exists(ppi_file):
        ppi_df = pd.read_csv(ppi_file, sep='\t', usecols=['gene1', 'gene2', 'combined_score'])
        ppi_df['gene1'] = ppi_df['gene1'].str.upper().str.strip()
        ppi_df['gene2'] = ppi_df['gene2'].str.upper().str.strip()
        filtered = ppi_df[
            ppi_df['gene1'].isin(cohort_gene_ID) &
            ppi_df['gene2'].isin(cohort_gene_ID) &
            (ppi_df['combined_score'] >= 400)
        ]
        ppi_edges = np.stack([
            filtered['gene1'].map(cohort_gene_ID).to_numpy(),
            filtered['gene2'].map(cohort_gene_ID).to_numpy(),
        ], axis=1).tolist()
        del ppi_df, filtered
        gc.collect()

    random.seed(42)
    shuffled = list(cohort_genes)
    random.shuffle(shuffled)
    n_val = int(n_genes * 0.15)
    n_test = int(n_genes * 0.15)
    val_g = set(shuffled[:n_val])
    test_g = set(shuffled[n_val:n_val + n_test])
    train_g = set(shuffled[n_val + n_test:])

    sl['label'] = 1.0
    non_sl['label'] = 0.0
    df = pd.concat([sl, non_sl], ignore_index=True)
    df['id1'] = df['gene1'].map(cohort_gene_ID)
    df['id2'] = df['gene2'].map(cohort_gene_ID)
    df = df.dropna(subset=['id1', 'id2']).copy()
    df['id1'] = df['id1'].astype(int)
    df['id2'] = df['id2'].astype(int)

    train_mask = df['gene1'].isin(train_g) & df['gene2'].isin(train_g)
    val_mask = (df['gene1'].isin(val_g) | df['gene2'].isin(val_g)) & ~(df['gene1'].isin(test_g) | df['gene2'].isin(test_g))
    test_mask = df['gene1'].isin(test_g) | df['gene2'].isin(test_g)
    train_df, val_df, test_df = df[train_mask], df[val_mask], df[test_mask]

    train_pairs = torch.tensor(train_df[['id1', 'id2']].to_numpy(), dtype=torch.long)
    train_labels = torch.tensor(train_df['label'].to_numpy(), dtype=torch.float)
    val_pairs = torch.tensor(val_df[['id1', 'id2']].to_numpy(), dtype=torch.long)
    val_labels = torch.tensor(val_df['label'].to_numpy(), dtype=torch.float)
    test_pairs = torch.tensor(test_df[['id1', 'id2']].to_numpy(), dtype=torch.long)
    test_labels = torch.tensor(test_df['label'].to_numpy(), dtype=torch.float)

    train_ids = [cohort_gene_ID[g] for g in train_g]
    train_mean = raw_x[train_ids].mean(dim=0)
    train_std = raw_x[train_ids].std(dim=0) + 1e-6
    scaled_x = (raw_x - train_mean) / train_std
    degrees = torch.zeros((n_genes, 1), dtype=torch.float32)
    if ppi_edges:
        edges_np = np.array(ppi_edges)
        np.add.at(degrees.numpy(), edges_np[:, 0], 1.0)
        np.add.at(degrees.numpy(), edges_np[:, 1], 1.0)
    X = torch.cat([scaled_x, degrees], dim=1)
    del raw_x, scaled_x
    gc.collect()

    train_id_set = set(cohort_gene_ID[g] for g in train_g)
    edges_np = np.array(ppi_edges) if ppi_edges else np.zeros((0, 2), dtype=int)
    if len(edges_np) > 0:
        full_edge_index = torch.tensor(ppi_edges, dtype=torch.long).t().contiguous()
        train_edge_mask = (
            np.isin(edges_np[:, 0], list(train_id_set)) &
            np.isin(edges_np[:, 1], list(train_id_set))
        )
        train_edges = edges_np[train_edge_mask].tolist()
        train_edge_index = (
            torch.tensor(train_edges, dtype=torch.long).t().contiguous()
            if train_edges else torch.zeros((2, 0), dtype=torch.long)
        )
    else:
        full_edge_index = train_edge_index = torch.zeros((2, 0), dtype=torch.long)

    data_train = Data(x=X, edge_index=train_edge_index)
    data_full = Data(x=X, edge_index=full_edge_index)
    num_features = X.shape[1]

    seed_aucs = []
    for seed in range(n_seeds):
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        gat_model = GATEncoder(in_chan=num_features, hid_chan=64, out_chan=32)
        regressor = PairRegressor(embed_dim=32)
        best_state, _ = train_single_model(
            gat_model, regressor, data_train, data_full,
            train_pairs, train_labels, val_pairs, val_labels, epochs=120, lr=0.005
        )
        if best_state:
            gat_model.load_state_dict(best_state['gat'])
            regressor.load_state_dict(best_state['reg'])
        gat_model.eval()
        regressor.eval()
        with torch.no_grad():
            embeddings = gat_model(data_full.x, data_full.edge_index)
            probs = torch.sigmoid(regressor(embeddings, test_pairs)).cpu().numpy()
        auc = roc_auc_score(test_labels.cpu().numpy(), probs)
        seed_aucs.append(auc)
        print(f"  seed {seed}: AUC = {auc:.4f}")
        del gat_model, regressor, embeddings, probs
        gc.collect()

    seed_aucs = np.array(seed_aucs)
    print(f"{cohort_name} ensemble AUC: {seed_aucs.mean():.4f} +/- {seed_aucs.std():.4f}")
    return seed_aucs


metabric_aucs = run_hydra_on_cohort(METABRIC_PATH, 'METABRIC')

# Compare against TCGA's ppi_only seeds from ablation cell
if 'results_rows' in dir():
    tcga_ppi_rows = [r for r in results_rows if r['graph_mode'] == 'ppi_only']
    if tcga_ppi_rows:
        tcga_aucs = np.array(tcga_ppi_rows[0]['seed_aucs'])
        t_stat, p_val = stats.ttest_ind(tcga_aucs, metabric_aucs)
        print(f"\nTCGA (ppi_only, ablation cell): {tcga_aucs.mean():.4f} +/- {tcga_aucs.std():.4f}")
        print(f"METABRIC:                       {metabric_aucs.mean():.4f} +/- {metabric_aucs.std():.4f}")
        print(f"Independent samples t-test: t={t_stat:.4f}, p={p_val:.4f}")
    else:
        print("\nppi_only results not found in results_rows")
else:
    print("\nresults_rows not in scope (ablation cell not yet run) "
          "skipping TCGA comparison; METABRIC AUC reported above stands alone.")

In [ ]:
def split_membership(gene):
    if gene in train_genes:
        return 'train'
    elif gene in val_genes:
        return 'val'
    elif gene in test_genes:
        return 'test'
    else:
        return 'not in vocabulary'


print("VEGFA:", split_membership('VEGFA'))
for gene in ['DAPK2', 'FHIT', 'EXT1', 'MNX1', 'CD22', 'ATM', 'SMO', 'BAX', 'IGF1R', 'CSF1R']:
    print(f"  {gene}: {split_membership(gene)}")

print("\nPTPRD:", split_membership('PTPRD'))
print("  VEGFA (as PTPRD's partner):", split_membership('VEGFA'))

print("\nSTARD9:", split_membership('STARD9'))
for gene in ['MLH1', 'MSH2', 'MSH6', 'PMS2']:
    print(f"  {gene}: {split_membership(gene)}")

# Precision@K

In [ ]:
"""
Precision@K for HYDRA (ppi-only, sl-edges) vs. DDGCN, gene-level inductive
evaluation.
"""
import numpy as np
import torch
from scipy import stats as scipy_stats
import matplotlib.pyplot as plt

k_vals = [10, 25, 50, 100, 200, 500, 1000]


def pk_from_probs(probs, labels, tie_break_seed=0):
    """Precision@K with randomized tie-breaking."""
    probs = np.asarray(probs)
    n = len(probs)
    rng = np.random.RandomState(tie_break_seed)
    jitter = rng.uniform(low=-1e-9, high=1e-9, size=n)
    order = np.argsort(probs + jitter)[::-1]
    sorted_labels = labels[order]
    return [sorted_labels[:k].mean() for k in k_vals]


def pk_curve_from_ensemble(gat_list, reg_list, data_full, test_pairs, test_labels_np, label):
    """Inference-only per-seed P@K over an already-trained ensemble."""
    seed_pks, seed_aucs = [], []
    for seed_idx, (gat, reg) in enumerate(zip(gat_list, reg_list)):
        gat.eval()
        reg.eval()
        with torch.no_grad():
            embeddings = gat(data_full.x, data_full.edge_index)
            probs = torch.sigmoid(reg(embeddings, test_pairs)).cpu().numpy()
        pk = pk_from_probs(probs, test_labels_np, tie_break_seed=seed_idx)
        seed_pks.append(pk)
        seed_aucs.append(pk[k_vals.index(100)])
        print(f"  [{label}] seed {seed_idx}: P@100={pk[k_vals.index(100)]:.4f}")
    arr = np.array(seed_pks)
    return arr.mean(axis=0), 2 * scipy_stats.sem(arr, axis=0)


test_labels_np = test_labels.cpu().numpy()
test_pairs_np = test_pairs.cpu().numpy()

# ppi only and sl only hydra

ppi_mean = ppi_2sem = sl_mean = sl_2sem = None

if 'saved_models' in dir() and 'ppi_only' in saved_models and len(saved_models['ppi_only'][0]) >= 5:
    gat_list, reg_list, data_full = saved_models['ppi_only']
    print("HYDRA ppi-only (5-seed ensemble, inference only):")
    ppi_mean, ppi_2sem = pk_curve_from_ensemble(gat_list, reg_list, data_full, test_pairs, test_labels_np, 'ppi-only')
else:
    print("saved_models['ppi_only'] not found or has fewer than 5 seeds -- "
          "run the main training cell (and/or the ablation cell) first. Skipping this curve.")

if 'saved_models' in dir() and 'sl_edges' in saved_models and len(saved_models['sl_edges'][0]) >= 5:
    gat_list, reg_list, data_full = saved_models['sl_edges']
    print("\nHYDRA sl-edges (5-seed ensemble, inference only):")
    sl_mean, sl_2sem = pk_curve_from_ensemble(gat_list, reg_list, data_full, test_pairs, test_labels_np, 'sl-edges')
else:
    print("\nsaved_models['sl_edges'] not found or has fewer than 5 seeds -- "
          "run the three-condition ablation cell first. Skipping this curve.")

# DDGCN

print("\nDDGCN P@K (5-seed):")
if 'ddgcn_tst_score_seed' not in dir():
    raise RuntimeError("ddgcn_tst_score_seed not in scope -- run the DDGCN cell first "
                        "(with the per-seed capture in its inductive loop).")

ddgcn_seed_pks = [
    pk_from_probs(scores, inductive_test_labels, tie_break_seed=i)
    for i, scores in enumerate(ddgcn_tst_score_seed)
]
for i, pk in enumerate(ddgcn_seed_pks):
    print(f"  [DDGCN] seed {i}: P@100={pk[k_vals.index(100)]:.4f}")
ddgcn_pk_arr = np.array(ddgcn_seed_pks)
ddgcn_pk = ddgcn_pk_arr.mean(axis=0)
ddgcn_2sem = 2 * scipy_stats.sem(ddgcn_pk_arr, axis=0)
print(f"DDGCN P@K (mean): {[round(p, 4) for p in ddgcn_pk]}")

# Plot

%matplotlib inline
plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 10,
    'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#dddddd', 'grid.linewidth': 0.5,
    'grid.linestyle': '--', 'legend.frameon': True, 'legend.edgecolor': '#aaaaaa',
})
BLUE = '#2166ac'
GREEN = '#4dac26'
RED = '#d6604d'
GRAY = '#888888'
EKW = dict(elinewidth=1.0, capsize=4, capthick=1.0, ecolor='black')

fig, ax = plt.subplots(figsize=(5, 3.8))

if ppi_mean is not None:
    ax.errorbar(k_vals, ppi_mean, yerr=ppi_2sem,
                color=BLUE, marker='o', markersize=6, linewidth=1.5,
                markerfacecolor='white', markeredgewidth=1.5,
                label='HYDRA ppi-only (±2 SEM, n=5)', **EKW)
if sl_mean is not None:
    ax.errorbar(k_vals, sl_mean, yerr=sl_2sem,
                color=GREEN, marker='^', markersize=6, linewidth=1.5,
                markerfacecolor='white', markeredgewidth=1.5,
                label='HYDRA sl-edges (±2 SEM, n=5)', **EKW)

ax.errorbar(k_vals, ddgcn_pk, yerr=ddgcn_2sem,
            color=RED, marker='s', markersize=6, linewidth=1.5,
            markerfacecolor='white', markeredgewidth=1.5,
            label='DDGCN inductive (±2 SEM, n=5)', **EKW)

ax.plot(k_vals, [0.4996] * len(k_vals),
        color=GRAY, linestyle='--', linewidth=1.0,
        label='Random baseline')

ax.set_xscale('log')
ax.set_xticks(k_vals)
ax.set_xticklabels([str(k) for k in k_vals], fontsize=9)
ax.set_xlabel('K')
ax.set_ylabel('Precision@K')
ax.set_ylim(0.15, 1.05)
ax.set_title('Precision@K: HYDRA vs DDGCN\n(gene-level inductive evaluation)', fontsize=10)
ax.legend(loc='lower right', fontsize=8)
fig.tight_layout()
plt.show()
fig.savefig('FigP@k.tif', dpi=300, bbox_inches='tight', format='tiff')
fig.savefig('FigP@k.png', dpi=300, bbox_inches='tight')

from google.colab import files
files.download("FigP@k.tif")
files.download("FigP@k.png")